**ElectricityMaps Data**
First of all, we need to get the data from ElectricityMaps. We use their API to fetch the data. A big advantage of using their API was that we could get real-time data, which would crucial if we wanted to implement our reinforcement learning model in real-time. We will be using the `requests` library in Python to make HTTP requests to the API.

There is barely any need to do any data cleaning and we'll leave most of the preprocessing to the model. However, we will need to convert the data into a format that can be easily fed into the model. We will convert the data into a pandas DataFrame, which is a powerful data structure for data manipulation and analysis in Python.

We start by installing the necessary libraries from the requirements.txt file and then import them.

In [1]:
pip install -r -q requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '-q'


In [2]:
import requests
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pandas as pd
import seaborn as sns

Setting up the API call to fetch data from ElectricityMaps. We specifiy the date range and the location for which we want to fetch the data. We also need to provide our API key for authentication.

We want to data in an hourly temporal granularity however we're only allowed to fetch data 10 days at a time, so we need to loop through the date range and fetch the data in chunks. We will store the fetched data in a list and then concatenate it into a single DataFrame.

In [3]:
API_KEY = "X6mXbkbMh9Y5HDvPhgPp"


headers = {
    "auth-token": API_KEY
}
dateRange = {"start": "2022-01-01T00:00Z", "end": "2025-01-01T00:00Z"}

start_dt = pd.to_datetime(dateRange["start"], utc=True)
end_dt = pd.to_datetime(dateRange["end"], utc=True)
chunk = pd.Timedelta(days=10)
zones = ["DK-DK1", "DK-DK2"]
date_ranges = []
current_start = start_dt
while current_start < end_dt:
    current_end = min(current_start + chunk, end_dt)
    for zone in zones:
        date_ranges.append(
        (   
            zone,
            current_start.strftime("%Y-%m-%dT%H:%MZ"),
            current_end.strftime("%Y-%m-%dT%H:%MZ"),
        )
    )
    current_start = current_end

Fetching the day-ahead spot price.

In [4]:
PDA_df = pd.DataFrame()
url = "https://api.electricitymaps.com/v4/price-day-ahead/past-range"

for zone, start, end in date_ranges:
    params = {
        "zone": zone,
        "start": start,
        "end": end,
        "temporalGranularity": "hourly",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    #print(response.status_code)
    #print(response.text)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["data"])
    PDA_df = pd.concat([PDA_df, df], ignore_index=True)

PDA_df["datetime"] = pd.to_datetime(PDA_df["datetime"])
PDA_df = PDA_df.set_index("datetime")

#print(PDA_df.head())

Fetching the electricity mix.

In [5]:
url = "https://api.electricitymaps.com/v4/electricity-mix/past-range"
MIX_df = pd.DataFrame()
for zone, start, end in date_ranges:
    params = {
        "zone": zone,
        "start": start,
        "end": end,
        "temporalGranularity": "hourly",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    #print(response.status_code)
    #print(response.text)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["data"])
    df["zone"] = zone
    MIX_df = pd.concat([MIX_df, df], ignore_index=True)


MIX_df["datetime"] = pd.to_datetime(MIX_df["datetime"])
MIX_df = MIX_df.set_index("datetime")



Fetching the electiricity flow data. It's data on the electricity flow between different regions. This data is important because it can give us insights into the supply and demand dynamics in different regions, which can affect the electricity prices.

In [6]:
url = "https://api.electricitymaps.com/v4/electricity-flows/past-range"
EF_df = pd.DataFrame()
for zone, start, end in date_ranges:
    params = {
        "zone": zone,
        "start": start,
        "end": end,
        "temporalGranularity": "hourly",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    #print(response.status_code)
    #print(response.text)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["data"])
    df["zone"] = zone
    EF_df = pd.concat([EF_df, df], ignore_index=True)


EF_df["datetime"] = pd.to_datetime(EF_df["datetime"])
EF_df = EF_df.set_index("datetime")



Fetching the carbon intensity data. 

In [7]:
url = "https://api.electricitymaps.com/v4/carbon-intensity/past-range"
CI_df = pd.DataFrame()
for zone, start, end in date_ranges:
    params = {
        "zone": zone,
        "start": start,
        "end": end,
        "temporalGranularity": "hourly",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    #print(response.status_code)
    #print(response.text)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["data"])
    CI_df = pd.concat([CI_df, df], ignore_index=True)


CI_df["datetime"] = pd.to_datetime(CI_df["datetime"])
CI_df = CI_df.set_index("datetime")



Fetching the total load in the specific zone. 

In [8]:
url = "https://api.electricitymaps.com/v4/total-reported-load/past-range"
TL_df = pd.DataFrame()
for zone, start, end in date_ranges:
    params = {
        "zone": zone,
        "start": start,
        "end": end,
        "temporalGranularity": "hourly",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    #print(response.status_code)
    #print(response.text)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["data"])
    TL_df = pd.concat([TL_df, df], ignore_index=True)


TL_df["datetime"] = pd.to_datetime(TL_df["datetime"])
TL_df = TL_df.set_index("datetime")



Merging all the data into a single DataFrame. We will merge the data on the timestamp and the zone, which are common columns in all the datasets. This will give us a comprehensive dataset that contains all the relevant information for our analysis and modeling. 

We'll also do some feature engineering adding time-based features such as the hour of the day, day of the week, and month of the year. These features can help the model capture temporal patterns in the data, which can be important for predicting electricity prices. 

We'll also drop some columns that we won't be using in our analysis and modeling. This will help reduce the dimensionality of the data and make it easier for the model to learn from the data. 

In [9]:
merged_df = pd.merge(PDA_df, EF_df, on=["datetime","zone"], suffixes=('_price', '_flow'))
merged_df = pd.merge(merged_df, MIX_df, on=["datetime","zone"], suffixes=('_merged', '_mix'))
merged_df = pd.merge(merged_df, TL_df, on=["datetime","zone"], suffixes=('_spot','_total_load'))
merged_df = merged_df.drop(columns=['createdAt_spot','updatedAt_spot','createdAt_total_load','updatedAt_total_load', 'isEstimated_spot','estimationMethod_spot','estimationMethod_total_load', 'isEstimated_total_load'])
merged_df = pd.merge(merged_df, CI_df, on=["datetime","zone"], suffixes=('_prev', '_CI'))
merged_df = merged_df.drop(columns=["isEstimated",'updatedAt_price', 'updatedAt','createdAt','estimationMethod', 'updatedAt_flow','updatedAt_flow','source_spot','source_total_load'])

#add day of week, month, hour of day, and year as features
merged_df['day_of_week'] = merged_df.index.dayofweek
merged_df['month'] = merged_df.index.month
merged_df['hour_of_day'] = merged_df.index.hour
merged_df['year'] = merged_df.index.year


Oil price data from FRED. We will be using the Brent Europe oil price, which is a widely used benchmark for oil prices. We will fetch the data from the FRED API and convert it into a pandas DataFrame. We will also do some preprocessing to make sure the data is in the right format for our analysis and modeling.

Unfortunately oil prices and gas prices are only available on a daily basis, so we need to forward fill the data to match the hourly granularity of our electricity data. This means that we will fill the missing values with the last available value. This is a common technique for handling missing data in time series analysis. 

Furthermore, oil and gas futures are only available on weekdays, so we need to forward fill the data for the weekends as well. This will ensure that we have a complete dataset with no missing values, which is important for training our model effectively. 

In [10]:
#Get oil price data from 

# FRED CSV endpoint for Brent Europe
FRED_ID = ["DCOILBRENTEU"]
for id in FRED_ID:
    url_brent = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id={id}"
        "&cosd=2021-12-31"
        "&coed=2025-01-01"
    )
    if id == "DCOILBRENTEU":
        brent = pd.read_csv(url_brent)
        brent.set_index("observation_date", inplace=True)
        brent = brent.rename(columns={"DCOILBRENTEU": "Oil_Price"})
        brent.index = pd.to_datetime(brent.index, utc=True)
        brent = brent.resample("D").ffill()
        brent.drop("2021-12-31", inplace=True)
        brent.drop("2025-01-01", inplace=True)
        brent.rename_axis("datetime", inplace=True)

brent = brent.resample("D").ffill().bfill()
print(brent.head())
brent.info()

                           Oil_Price
datetime                            
2022-01-01 00:00:00+00:00      77.24
2022-01-02 00:00:00+00:00      77.24
2022-01-03 00:00:00+00:00      78.25
2022-01-04 00:00:00+00:00      79.39
2022-01-05 00:00:00+00:00      80.60
<class 'pandas.DataFrame'>
DatetimeIndex: 1096 entries, 2022-01-01 00:00:00+00:00 to 2024-12-31 00:00:00+00:00
Freq: D
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Oil_Price  1096 non-null   float64
dtypes: float64(1)
memory usage: 17.1 KB


Gas price could also be relevant, but gas is a traded commdodity. Therefore we use Dutch natural gas price futures as a proxy for the gas price, we will only look at the close price. The gas price data is fetched from yahoo finance using the `yfinance` library in Python. We will fetch the data for the natural gas price futures and convert it into a pandas DataFrame. We will also do some preprocessing to make sure the data is in the right format for our analysis and modeling.

In [11]:
import yfinance as yf

gas = yf.download("TTF=F", start="2021-12-31", end="2025-01-01")
gas.head()

# remove multi-index
gas.columns = gas.columns.get_level_values(0)

# keep only close price
gas = gas[["Close"]]

# rename
gas = gas.rename(columns={"Close": "Gas_price"})

print(gas.head())

# make sure index is datetime
gas.index = pd.to_datetime(gas.index, utc=True)

# create full daily index
gas = gas.asfreq("D")

# forward fill weekends & holidays
gas["Gas_price"] = gas["Gas_price"].ffill()

gas.drop("2021-12-31", inplace=True)
gas.tail()

gas.rename_axis("datetime", inplace=True)

gas.info()

[*********************100%***********************]  1 of 1 completed

Price       Gas_price
Date                 
2021-12-31  70.343002
2022-01-03  80.433998
2022-01-04  88.740997
2022-01-05  91.522003
2022-01-06  96.501999
<class 'pandas.DataFrame'>
DatetimeIndex: 1096 entries, 2022-01-01 00:00:00+00:00 to 2024-12-31 00:00:00+00:00
Freq: D
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Gas_price  1096 non-null   float64
dtypes: float64(1)
memory usage: 17.1 KB


In [12]:
gas_oil = pd.merge(gas, brent, on=["datetime"], how="inner")
gas_oil=gas_oil.resample("h").ffill()
gas_oil.head()
gas_oil.rename(columns={"Date": "datetime"}, inplace=True)

In [13]:
merged_df = pd.merge(merged_df, gas_oil, on=["datetime"], how="inner")
merged_df.to_csv("final_data.csv")
merged_df = pd.read_csv("final_data.csv", index_col=0, parse_dates=True)

# Collecting Weather Data

This notebook collects the needed weather data for the project. It uses open-meteo.com API to get the weather data for the specified location and time period.


In [14]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

Then we set up the necessary parameters for the API call, such as the latitude and longitude of the location, the start and end dates for the data collection, and the weather parameters we want to collect.

For each electricity zone, we have found an approximate midpoint to collect the weather data. 
For DK1 (western Denmark) we use Silkeborg (55.4426, 11.7901) and for DK2 (eastern Denmark) we use Ringsted (56.1697, 9.5451).

In [15]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


locations = [
    {"name": "Ringsted", "latitude": 56.1697, "longitude": 9.5451},
    {"name": "Silkeborg", "latitude": 55.4426, "longitude": 11.7901},
    {"name": "Danmark", "latitude": 56, "longitude": 10}
]



# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": [place["latitude"] for place in locations],
	"longitude": [place["longitude"] for place in locations],
	"start_date": "2022-01-01",
	"end_date": "2024-12-31",
	"hourly": ["wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_speed_180m", "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_direction_180m", "uv_index", "sunshine_duration"],
}



Then the api is called, the data from the call is stored in a dataframe and the dataframe is saved as a csv file. The csv file is then used in the other notebooks.

In [19]:

responses = openmeteo.weather_api(url, params = params)

# Process 3 locations
for i, response in enumerate(responses):
	location = locations[i]
	print(f"\nLocation: {location['name']}")
	assert abs(response.Latitude() - location["latitude"]) < 0.01, "Latitude does not match"
	assert abs(response.Longitude() - location["longitude"]) < 0.01, "Longitude does not match"

	print(f"\nCoordinates: {response.Latitude()}°N {response.Longitude()}°E")
	#print(f"Elevation: {response.Elevation()} m asl")
	#print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_wind_speed_10m = hourly.Variables(0).ValuesAsNumpy()
	hourly_wind_speed_80m = hourly.Variables(1).ValuesAsNumpy()
	hourly_wind_speed_120m = hourly.Variables(2).ValuesAsNumpy()
	hourly_wind_speed_180m = hourly.Variables(3).ValuesAsNumpy()
	hourly_wind_direction_10m = hourly.Variables(4).ValuesAsNumpy()
	hourly_wind_direction_80m = hourly.Variables(5).ValuesAsNumpy()
	hourly_wind_direction_120m = hourly.Variables(6).ValuesAsNumpy()
	hourly_wind_direction_180m = hourly.Variables(7).ValuesAsNumpy()
	hourly_uv_index = hourly.Variables(8).ValuesAsNumpy()
	hourly_sunshine_duration = hourly.Variables(9).ValuesAsNumpy()
	
	hourly_data = {"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)}
	
	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
	hourly_data["wind_speed_120m"] = hourly_wind_speed_120m
	hourly_data["wind_speed_180m"] = hourly_wind_speed_180m
	hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
	hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
	hourly_data["wind_direction_120m"] = hourly_wind_direction_120m
	hourly_data["wind_direction_180m"] = hourly_wind_direction_180m
	hourly_data["uv_index"] = hourly_uv_index
	hourly_data["sunshine_duration"] = hourly_sunshine_duration
	
	hourly_dataframe = pd.DataFrame(data = hourly_data)
	#print("\nHourly data\n", hourly_dataframe)
	hourly_dataframe.to_csv(f"hourly_data_{location['name']}_{response.Latitude()}_{response.Longitude()}.csv", index = False)
	


Location: Ringsted

Coordinates: 56.1740608215332°N 9.545852661132812°E

Location: Silkeborg

Coordinates: 55.444580078125°N 11.78314208984375°E

Location: Danmark

Coordinates: 55.99775695800781°N 10.0052490234375°E


# Collecting Demand Data
Our demand data is collected from a kaggle dataset: https://www.kaggle.com/datasets/csafrit2/steel-industry-energy-consumption

This is a dataset of energy consumption for a steel factory called DAEWOO Steel Co. Ltd in Gwangyang, South Korea. The dataset contains energy consumption data from January 1, 2018 to December 31, 2018 for every quarter-hour.


We use the kagglehub library to download the latest version of the dataset and save it to the data directory. The path to the dataset files is printed for reference.

In [17]:
import kagglehub

# Download latest version to this directory
path = kagglehub.dataset_download("csafrit2/steel-industry-energy-consumption", output_dir="data", force_download= True)

c:\Users\anton\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 484k/484k [00:00<00:00, 798kB/s]

Extracting files...


Now we load the dataset into a dataframe and do some basic preprocessing to get it ready for analysis. We convert the date column to a datetime format and extract the week, day of the week, and hour of the day as new columns in the dataframe. This will allow us to analyze the demand patterns based on time.

In [18]:
import pandas as pd
import os.path

file = "steel_industry_data.csv"
folder = "data"
path = os.path.join(folder, file)
# Load the dataset into a dataframe
demand_df = pd.read_csv(path)

demand_df['date'] = pd.to_datetime(demand_df['date'], format="%d/%m/%Y %H:%M")

# Set the date column as the index
demand_df.set_index('date', inplace=True)


# reduce to hourly data by taking the sum of every 4 rows (since the original data is in 15-minute intervals)
demand_df = demand_df.resample('h').sum()

demand_df.to_csv("steel_industry_hourly_usage.csv", index = True)
